# PRISM Training Notebook
Interactive copy of the training shim for SLM LoRA fine-tuning. Configure cells as needed before launching a run.

## Prerequisites
- Ensure the PRISM repo (and SPINE dependency) are on `PYTHONPATH` or installed editable.
- Activate the same conda env used for CLI runs (`/home/jporras/miniconda3/envs/base3_10`).
- Log into Weights & Biases (`wandb login`) ahead of time.

In [ ]:
# Optional: set environment variables for logging
import os
os.environ.setdefault("WANDB_PROJECT", "SLM-distill-GNN")
os.environ.setdefault("WANDB_ENTITY", "alelab")
os.environ.setdefault("UNSLOTH_RETURN_LOGITS", "1")

'1'

## Dependencies

In [ ]:
# Core dependencies
import json
from dataclasses import dataclass, field
from typing import List

import wandb
from datasets import load_dataset
from transformers import HfArgumentParser
from trl import SFTConfig, SFTTrainer

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

from prism.eval.callbacks import EvalCallback
from prism.eval.run_eval import EvalSample
from prism.training.utils import (
    TurnAwareCollator,
    get_formatting_prompts_func,
    train_on_responses_only,
    )

/home/jporras/miniconda3/envs/prism/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Training Config

In [ ]:
@dataclass
class TrainConfig:
    name: str
    checkpoint_dir: str
    data: str
    bit4: bool = False
    eval_data: str = "../data/eval/eval_1_multi_step.json"
    r: int = 16
    base_model: str = "unsloth/Llama-3.2-3B-Instruct"
    wandb_project: str = "SLM-distill-GNN"
    epochs: int = 2
    val_frac: float = 0.1
    lora_alpha: int = 16
    lora_dropout: float = 0.2
    target_modules: List[str] = field(default_factory=lambda: [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ])
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    report_to: str = "wandb"
    learning_rate: float = 2e-4
    warmup_steps: int = 5
    weight_decay: float = 0.05
    debug: bool = True


## Manually exploring the model

In [ ]:
pretrained_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-bnb-4bit",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=False,
    )
model_to_tune = FastLanguageModel.get_peft_model(
    pretrained_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.2,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 2. Max memory: 23.677 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.2.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.19 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
# Example configuration template (edit and execute)
# PRISM COnfig
cfg = TrainConfig(
    name="sft_llama32_3b_4bit",
    checkpoint_dir="checkpoints/sft_llama32_3b_4bit",
    data="data/gen/spine_exp1/formatted.json",
    eval_data="data/eval/eval_1_multi_step.json",
    base_model="unsloth/Llama-3.2-1B-bnb-4bit",
    bit4=True,
    epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    r=16,
    lora_alpha=16,
    lora_dropout=0.2,
    debug=True,
)
cfg

TrainConfig(name='sft_llama32_3b_4bit', checkpoint_dir='checkpoints/sft_llama32_3b_4bit', data='data/gen/spine_exp1/formatted.json', bit4=True, eval_data='data/eval/eval_1_multi_step.json', r=16, base_model='unsloth/Llama-3.2-1B-bnb-4bit', wandb_project='SLM-distill-GNN', epochs=2, val_frac=0.1, lora_alpha=16, lora_dropout=0.2, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=2, report_to='wandb', learning_rate=0.0002, warmup_steps=5, weight_decay=0.05, debug=True)

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [ ]:
#tokenizer

In [ ]:
full_dataset = load_dataset("json", data_files=[cfg.data], split="train")
# Preprocessing
full_dataset = standardize_sharegpt(full_dataset)
formatting_prompts_func = get_formatting_prompts_func(tokenizer) # prism util
full_dataset = full_dataset.map(formatting_prompts_func, batched=True)
if "conversations" in full_dataset.column_names:
    full_dataset = full_dataset.remove_columns(["conversations"])
if cfg.val_frac > 0:
    dataset_size = len(full_dataset)
    val_size = int(dataset_size * cfg.val_frac)
    train_size = dataset_size - val_size
    train_val_split = full_dataset.train_test_split(
        test_size=val_size,
        train_size=train_size,
        seed=3407,
    )
    train_dataset = train_val_split["train"]
    val_dataset = train_val_split["test"]
    print(f"Dataset split: {train_size} training samples, {val_size} validation samples")
else:
    train_dataset = full_dataset
    val_dataset = None
    print(f"Using all {len(full_dataset)} samples for training (no validation)")


Map: 100%|██████████| 990/990 [00:00<00:00, 11893.86 examples/s]

Dataset split: 891 training samples, 99 validation samples


In [ ]:
train_dataset.select(range(10)).map(lambda x: tokenizer(x["text"]))

Map: 100%|██████████| 10/10 [00:00<00:00, 468.00 examples/s]


Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 10
})

In [ ]:

trainer_kwargs = {
    "model": model_to_tune,
    "tokenizer": tokenizer,
    "train_dataset": train_dataset,
    "data_collator": TurnAwareCollator(
        tokenizer=tokenizer, padding="longest", include_turns=False
    ),
    "args": sft_config,
}
if val_dataset is not None:
    trainer_kwargs["eval_dataset"] = val_dataset

trainer = SFTTrainer(**trainer_kwargs)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

with open(cfg.eval_data) as f:
    eval_payload = json.load(f)
eval_samples = [
    EvalSample(
        task=entry["task"],
        answer=entry["answer"],
        graph=eval_payload["graph"],
        init_node=entry["init_node"],
    )
    for entry in eval_payload["tasks"]
]

trainer.add_callback(EvalCallback(eval_samples))

/tmp/ipykernel_3572253/1867432647.py:13: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(**trainer_kwargs)
Converting train dataset to ChatML (num_proc=48): 100%|██████████| 891/891 [00:00<00:00, 3309.83 examples/s]
Applying chat template to train dataset (num_proc=48): 100%|██████████| 891/891 [00:07<00:00, 124.96 examples/s]
Converting eval dataset to ChatML (num_proc=48): 100%|██████████| 99/99 [00:00<00:00, 373.32 examples/s]
Applying chat template to eval dataset (num_proc=48): 100%|██████████| 99/99 [00:07<00:00, 13.91 examples/s]
Map: 100%|██████████| 99/99 [00:00<00:00, 1459.44 examples/s]


NameError: name 'config' is not defined

In [ ]:
1

## Train model Function

In [ ]:
def train_model(config: TrainConfig):
    """Train via Unsloth LoRA SFT.
    Expects JSON data formatted like `data/gen/.../formatted.json`.
    """
    os.environ["WANDB_PROJECT"] = config.wandb_project
    os.environ.setdefault("WANDB_ENTITY", "alelab")
    os.environ.setdefault("UNSLOTH_RETURN_LOGITS", "1")

    save_name = config.name
    max_seq_length = 2048
    load_in_4bit = False
    dtype = None

    save_name += f"_r{config.r}"
    if config.bit4:
        save_name += "_4bit"
        load_in_4bit = True

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.base_model,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=config.r,
        target_modules=config.target_modules,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

    full_dataset = load_dataset("json", data_files=[config.data], split="train")
    if config.debug:
        full_dataset = full_dataset.select(range(min(100, len(full_dataset))))

    full_dataset = standardize_sharegpt(full_dataset)
    formatting_prompts_func = get_formatting_prompts_func(tokenizer) # prism util.
    full_dataset = full_dataset.map(formatting_prompts_func, batched=True)
    if "conversations" in full_dataset.column_names:
        full_dataset = full_dataset.remove_columns(["conversations"])

    if config.val_frac > 0:
        dataset_size = len(full_dataset)
        val_size = int(dataset_size * config.val_frac)
        train_size = dataset_size - val_size
        train_val_split = full_dataset.train_test_split(
            test_size=val_size,
            train_size=train_size,
            seed=3407,
        )
        train_dataset = train_val_split["train"]
        val_dataset = train_val_split["test"]
        print(f"Dataset split: {train_size} training samples, {val_size} validation samples")
    else:
        train_dataset = full_dataset
        val_dataset = None
        print(f"Using all {len(full_dataset)} samples for training (no validation)")

    sft_config = SFTConfig(
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        warmup_steps=config.warmup_steps,
        num_train_epochs=config.epochs,
        learning_rate=config.learning_rate,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=config.weight_decay,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to=config.report_to,
    )

    trainer_kwargs = {
        "model": model,
        "tokenizer": tokenizer,
        "train_dataset": train_dataset,
        "data_collator": TurnAwareCollator(
            tokenizer=tokenizer, padding="longest", include_turns=False
        ),
        "args": sft_config,
    }
    if val_dataset is not None:
        trainer_kwargs["eval_dataset"] = val_dataset

    trainer = SFTTrainer(**trainer_kwargs)
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
    )

    with open(config.eval_data) as f:
        eval_payload = json.load(f)
    eval_samples = [
        EvalSample(
            task=entry["task"],
            answer=entry["answer"],
            graph=eval_payload["graph"],
            init_node=entry["init_node"],
        )
        for entry in eval_payload["tasks"]
    ]

    trainer.add_callback(EvalCallback(eval_samples))
    return trainer


In [ ]:
# Example configuration template (edit and execute)
cfg = TrainConfig(
    name="sft_llama32_3b_4bit",
    checkpoint_dir="checkpoints/sft_llama32_3b_4bit",
    data="data/gen/spine_exp1/formatted.json",
    eval_data="data/eval/eval_1_multi_step.json",
    base_model="unsloth/Llama-3.2-1B-bnb-4bit",
    bit4=True,
    epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    r=16,
    lora_alpha=16,
    lora_dropout=0.2,
    debug=True,
)
cfg

In [ ]:
# TRAINING ENTRY POINT
# Uncomment the next line once ready to launch training
# trainer = train_model(cfg)
# trainer.train()


In [1]:
1

1